# Providers 00 - Auto Resolution

Objetivo: declarar providers y limites sin ejecutar modelos. `RuntimeConfig.describe()` hace observable la seleccion antes de construir Agents o Systems.

**Lugar en el modelo:** configuración de Runtime/Provider, antes de construir unidades de cómputo.

**Evidencia exigida:** `RuntimeConfig.describe()` debe mostrar la prioridad y la resolución en seco. Si OpenAI y Bedrock están listos a la vez, gana el primero que aparezca en `provider_priority`; un provider explícito no participa en esa competencia.

**Límite de la evidencia:** este capítulo no ejecuta inferencia; `configured` o `ready` todavía no significan `passed`.

## Parametros de la demostracion

| Parametro | Valor inicial | Que puedes cambiar |
|---|---|---|
| provider | python-runtime | OpenAI, vLLM o Bedrock mediante toolkit.runtime. |
| scheduler | limites locales | Timeout, retries, turnos y concurrencia. |
| profile | provider_profile | Inspeccion declarativa sin ejecutar modelos. |

## 1) Scheduler compartido

El scheduler expresa limites; no ejecuta trabajo por si mismo.

In [ ]:
import agentic_systems as toolkit

scheduler = toolkit.scheduler(
    timeout_s=60,
    max_retries=1,
    max_tool_calls=4,
    max_turns=4,
    max_concurrency=1,
)
toolkit.show_json(scheduler, title="SchedulerConfig")

## 2) Declarar rutas de provider

Crear un runtime es seguro y declarativo. Credenciales, endpoints y modelos se leen mediante defaults/configuracion; no se imprimen secretos.

In [ ]:
provider_names = ["python-runtime", "openai-runtime", "vllm-runtime", "bedrock-runtime"]
runtimes = {
    name: toolkit.runtime(provider=name, scheduler=scheduler)
    for name in provider_names
}
auto_runtime = toolkit.runtime(
    provider="auto",
    provider_priority=["openai-runtime", "vllm-runtime", "bedrock-runtime"],
    scheduler=scheduler,
)

toolkit.show_json(
    {name: runtime.describe() for name, runtime in runtimes.items()},
    title="Explicit runtime routes",
)
auto_description = auto_runtime.describe()
assert auto_description["mode"] == "auto"
assert auto_description["provider_priority"] == [
    "openai-runtime", "vllm-runtime", "bedrock-runtime"
]
selected_provider = auto_description.get("selected_provider")
if selected_provider is not None:
    assert selected_provider in auto_description["provider_priority"]

toolkit.show_json(auto_description, title="Auto runtime route")

## 3) Matriz declarada de capacidades

Los profiles describen soporte de contrato. No sustituyen una prueba live.

In [ ]:
provider_profiles = [
    profile.to_dict()
    for profile in toolkit.providers.provider_profiles()
]
toolkit.show_json(provider_profiles, title="Provider profiles")

## 4) API realmente ejercitada

In [ ]:
api_coverage = [
    "toolkit.scheduler",
    "toolkit.runtime(provider=...)",
    "RuntimeConfig.describe",
    "toolkit.providers.provider_profiles",
    "toolkit.show_json",
]
toolkit.show_json(api_coverage, title="Runtime API coverage")

## Resultado e interpretacion

Configuracion observable para cuatro providers y una ruta auto. Ninguna celda ejecuta inferencia.